# 05 — Train Words

## What this notebook does
Loads the sequences we recorded in `collect_data.py`, trains an LSTM neural network
to recognise words from sequences of hand landmarks, evaluates it, and saves the model.

## Why LSTM and not Random Forest like letters?
Letters are static — one frame tells you everything.
Words are motion — "hello" is a wave, "thanks" moves from chin outward.

A Random Forest looks at one snapshot.
An LSTM looks at a sequence of snapshots over time and learns the pattern of movement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)

## Step 1 — Load the recorded sequences

Each `.npy` file is one sample — a sequence of 30 frames, each frame being 63 landmark values.
We load all of them into `X` (the sequences) and `y` (the labels).

After loading, `X` will have shape `(total_samples, 30, 63)`:
- `total_samples` = number of recordings across all signs
- `30` = frames per sequence
- `63` = landmark values per frame

In [ ]:
DATA_DIR        = '../data/raw/my_signs'
SEQUENCE_LENGTH = 30

signs = sorted(os.listdir(DATA_DIR))
print(f"Signs found: {signs}")

X, y = [], []

for sign in signs:
    sign_path = os.path.join(DATA_DIR, sign)
    files     = os.listdir(sign_path)

    for file in files:
        sequence = np.load(os.path.join(sign_path, file))

        # Only use sequences with the right length
        if sequence.shape == (SEQUENCE_LENGTH, 63):
            X.append(sequence)
            y.append(sign)
        else:
            print(f"⚠ Skipped {sign}/{file} — shape {sequence.shape}")

X = np.array(X)
y = np.array(y)

print(f"\nX shape: {X.shape}   (samples, frames, landmarks)")
print(f"y shape: {y.shape}")
print(f"\nSamples per sign:")
for sign in signs:
    print(f"  {sign}: {np.sum(y == sign)}")

## Step 2 — Encode labels and split data

Same as with letters — we encode the sign names as numbers for the model.
Then we split 80% for training and 20% for testing.

One difference from the letter model: we use `to_categorical` to one-hot encode the labels.
This is needed for the LSTM's softmax output layer.

```
"hello"  →  0  →  [1, 0, 0, 0, 0, 0, 0]
"thanks" →  1  →  [0, 1, 0, 0, 0, 0, 0]
"yes"    →  2  →  [0, 0, 1, 0, 0, 0, 0]
...
```

In [ ]:
le        = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot  = to_categorical(y_encoded)

num_classes = len(signs)
print(f"Number of classes: {num_classes}")
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

## Step 3 — Build the LSTM model

The architecture:

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 63)),
    Dropout(0.3),
    LSTM(32, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 4 — Set up training callbacks

Two callbacks that make training smarter:

**EarlyStopping** — watches validation accuracy. If it stops improving for 15 epochs in a row,
training stops automatically. This prevents wasting time and overfitting.

**ReduceLROnPlateau** — if the model gets stuck, automatically reduces the learning rate
to take smaller steps. Often helps the model find a better solution.

`restore_best_weights=True` means we always end up with the best version of the model,
not the last one.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        verbose=1
    )
]

## Step 5 — Train the model

`epochs=100` means a maximum of 100 training passes through the data.
EarlyStopping will likely stop it well before that.

`validation_split=0.2` means during training we use 20% of the training data
to monitor how well the model generalises — this is separate from the test set.

Watch the `val_accuracy` number as it trains — that's the one that matters.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete")

## Step 6 — Plot training history

Two charts:
- **Accuracy** — training vs validation accuracy over epochs. Both should rise together.
- **Loss** — training vs validation loss over epochs. Both should fall together.

**Signs of a healthy training run:**
- Both curves move in the same direction
- Validation accuracy ends up above 80%

**Signs of overfitting:**
- Training accuracy keeps rising but validation accuracy plateaus or drops
- If you see this, we need more data or stronger dropout

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'],     label='Train accuracy')
ax1.plot(history.history['val_accuracy'], label='Val accuracy')
ax1.set_title('Model accuracy over epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'],     label='Train loss')
ax2.plot(history.history['val_loss'], label='Val loss')
ax2.set_title('Model loss over epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7 — Evaluate on the test set

Now we check performance on the 20% the model never saw at all — not during training,
not during validation. This is the true measure of how it will perform in the real world.

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred == y_true)
print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\nPer-sign breakdown:")
print(classification_report(y_true, y_pred, target_names=le.classes_))

## Step 8 — Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Confusion matrix — word predictions')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 9 — Save the model and label encoder

In [ ]:
os.makedirs('../models', exist_ok=True)

# Save LSTM model
model.save('../models/word_model.keras')

# Save label encoder
with open('../models/word_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("✅ Saved: models/word_model.keras")
print("✅ Saved: models/word_label_encoder.pkl")

## ✅ Notebook complete

**What we built:**
- Loaded 30-frame sequences recorded from your own camera
- Built an LSTM that reads motion over time, not just static positions
- Trained, evaluated, and saved the word model

**What's next:**
Update `src/run.py` to add a word mode — load the LSTM model, collect
30 frames in a rolling window, predict the word, and show it on screen.

That's Phase 2 done. After that is Phase 3 — scaling up to WLASL words.